# Word2Vec from scratch: skip-gram with full softmax

**No prior Word2Vec knowledge needed.** We will teach a small model to predict a nearby word. From `cat likes milk`, the center word `likes` has two observed neighbors, `cat` and `milk`. Each word gets a learned list of numbers called an **embedding**. Words used in similar settings can end up with similar embeddings.

This is an AI method because its parameters are **learned from examples by gradient descent**. We implement the training math with Python loops. NumPy appears only in the final independent check. The tiny built-in corpus keeps a complete run under one minute.

**Route:** create training pairs → initialize matrices → score words → softmax → loss → derivatives → repeated updates → experiment → NumPy check.

## 1. Sentences become labeled training pairs

A **token** is one word. A **vocabulary** is the set of distinct words, and its size is $V$. Each word gets an integer ID from $0$ to $V-1$.

Take the sentence `cat likes milk`. With a context window of radius $r=1$, `likes` is next to both `cat` and `milk`. We create the pairs `(likes, cat)` and `(likes, milk)`, along with `(cat, likes)` and `(milk, likes)`. We do not pair `cat` with `milk` because they are two positions apart.

$$
\mathcal D
= \left\{(w_i,w_j)\;\middle|\;0<|i-j|\le r,\quad
    i,j\text{ are in the same sentence}\right\}.
\tag{E1}
$$

In one pair $(c,o)\in\mathcal D$, $c$ is the **center** word and $o$ is the observed **outside** or context word. The task is to predict $o$ given $c$. The code in the next two cells constructs this pair list.

In [1]:
# E1: a small built-in corpus makes pair creation visible and reproducible.
sentences = [
    'cat likes milk', 'kitten likes milk', 'dog likes milk', 'puppy likes milk',
    'cat drinks milk', 'kitten drinks milk', 'dog drinks water', 'puppy drinks water',
    'cat chases mouse', 'kitten chases mouse', 'dog chases ball', 'puppy chases ball',
    'apple is sweet', 'banana is sweet', 'orange is sweet', 'pear is sweet',
    'apple tastes good', 'banana tastes good', 'orange tastes good', 'pear tastes good',
    'car drives road', 'truck drives road', 'bus drives road', 'van drives road',
    'car has wheels', 'truck has wheels', 'bus has wheels', 'van has wheels',
]
tokenized = [s.lower().split() for s in sentences]  # E1: one list of tokens per sentence.
vocab = sorted({w for s in tokenized for w in s})  # E1: distinct words.
word_to_id = {w: i for i, w in enumerate(vocab)}  # E1: word → integer ID.
id_to_word = {i: w for w, i in word_to_id.items()}
V = len(vocab)
print('sentences =', len(sentences), 'vocabulary size V =', V)
print('vocabulary =', vocab)

sentences = 28 vocabulary size V = 27
vocabulary = ['apple', 'ball', 'banana', 'bus', 'car', 'cat', 'chases', 'dog', 'drinks', 'drives', 'good', 'has', 'is', 'kitten', 'likes', 'milk', 'mouse', 'orange', 'pear', 'puppy', 'road', 'sweet', 'tastes', 'truck', 'van', 'water', 'wheels']


In [2]:
def make_pairs(sentences_as_words, word_to_id, radius=1):
    # E1: pair each center position i with every nearby position j.
    pairs = []
    for sentence in sentences_as_words:
        for i, center in enumerate(sentence):
            for j in range(max(0, i-radius), min(len(sentence), i+radius+1)):
                if i != j:
                    pairs.append((word_to_id[center], word_to_id[sentence[j]]))
    return pairs

pairs = make_pairs(tokenized, word_to_id, radius=1)  # E1: training data.
print('training pairs =', len(pairs))
print('first eight =', [(id_to_word[c], id_to_word[o]) for c, o in pairs[:8]])
assert len(pairs) == 4 * len(sentences)

training pairs = 112
first eight = [('cat', 'likes'), ('likes', 'cat'), ('likes', 'milk'), ('milk', 'likes'), ('kitten', 'likes'), ('likes', 'kitten'), ('likes', 'milk'), ('milk', 'likes')]


## 2. The two parameter matrices and their shapes

Choose an **embedding dimension** $d$, the number of coordinates used to represent one word. We learn two matrices because a word can appear as the center or as a possible context word:

| Symbol | Shape | Meaning |
|---|---:|---|
| $W_{\mathrm{in}}$ | $V\times d$ | One center vector per vocabulary word |
| $W_{\mathrm{out}}$ | $V\times d$ | One candidate-context vector per vocabulary word |
| $h_c$ | $d\times1$ | Vector of the current center word $c$ |
| $s(c,\cdot)$ | $V\times1$ | One score for each candidate context word |

For center-word ID $c$, take row $c$ from $W_{\mathrm{in}}$ and regard it as a column vector $h_c$. (Equivalently, multiply $W_{\mathrm{in}}^\mathsf{T}$ by a one-hot vector $x_c\in\mathbb R^V$.) For every candidate context-word ID $j$, calculate a **compatibility score**:

$$
\begin{aligned}
h_c &= W_{\mathrm{in}}[c,:]^\mathsf{T}
     = W_{\mathrm{in}}^\mathsf{T}x_c
     &&\in\mathbb R^d,\\
s(c,j) &= W_{\mathrm{out}}[j,:]h_c
       = \sum_{k=1}^{d} W_{\mathrm{out}}[j,k](h_c)_k,
     &&j=0,\ldots,V-1.
\end{aligned}
\tag{E2}
$$

Think of $s(c,j)$ as the model's answer to **“How well does candidate word $j$ fit near center word $c$?”** It is a dot product, so it combines the two word vectors coordinate by coordinate. A higher score means a better match *relative to other candidates*. A score is not yet a probability; it may be negative or greater than 1. The list of all $V$ scores is $s(c,\cdot)=W_{\mathrm{out}}h_c$.

The code selects the row directly instead of making a mostly-zero one-hot vector. Small random initial weights let different words learn different roles.

In [3]:
import math
import random

def initialize(V, d, seed=4):
    # E2: construct two V×d matrices with small random entries.
    rng = random.Random(seed)
    scale = 0.5 / d
    w_in = [[rng.uniform(-scale, scale) for _ in range(d)] for _ in range(V)]
    w_out = [[rng.uniform(-scale, scale) for _ in range(d)] for _ in range(V)]
    return w_in, w_out

example_in, example_out = initialize(V, d=8)
print('W_in shape =', len(example_in), '×', len(example_in[0]))
print('W_out shape =', len(example_out), '×', len(example_out[0]))
print('cat center vector =', [round(x, 4) for x in example_in[word_to_id['cat']]])

W_in shape = 27 × 8
W_out shape = 27 × 8
cat center vector = [0.001, -0.0108, 0.0124, -0.0086, -0.0423, -0.0244, 0.0391, -0.0571]


## 3. From compatibility scores to probabilities

For a given center word $c$, we have one score $s(c,j)$ for **each** candidate context word $j$. **Full softmax** turns those scores into probabilities that are positive and sum to 1:

$$
P(j\mid c)
=\frac{e^{s(c,j)}}{\sum_{t=0}^{V-1}e^{s(c,t)}}.
\tag{E3}
$$

The exponential makes every contribution positive. The denominator adds contributions from **all** vocabulary words, so the probabilities total 1. For intuition, imagine just three candidates with scores $2$, $1$, and $0$. Their probabilities are approximately $0.665$, $0.245$, and $0.090$; the highest-scoring candidate gets the highest probability.

If $o$ is the context word that actually appeared next to $c$, our **negative log-likelihood loss** is

$$
L(c,o)=-\log P(o\mid c).
\tag{E4}
$$

A high probability for the observed word gives a small loss. Before learning, probabilities are often near $1/V$, giving a loss near $\log V$.

We use the formula **without subtracting a maximum score** to keep the first implementation easy to read. It works for this tiny example. For a large model, $e^{s(c,j)}$ can overflow; a production implementation usually subtracts the largest score before exponentiating. That changes neither the mathematical probabilities nor the gradients.

In [4]:
def forward(center, w_in, w_out):
    h = w_in[center]  # E2: selected center vector, length d.
    scores = []
    for outside_vector in w_out:
        score = 0.0
        for k in range(len(h)):
            score += outside_vector[k] * h[k]  # E2: dot product.
        scores.append(score)
    exp_scores = [math.exp(score) for score in scores]  # E3: direct softmax numerator.
    total = sum(exp_scores)  # E3: denominator.
    probabilities = [value / total for value in exp_scores]  # E3: length V.
    return scores, probabilities

c, o = pairs[0]
_, p = forward(c, example_in, example_out)
print('pair =', id_to_word[c], '→', id_to_word[o])
print('P(observed outside word | center) =', round(p[o], 5))
print('loss =', round(-math.log(p[o]), 5))  # E4.
assert abs(sum(p)-1) < 1e-12

pair = cat → likes
P(observed outside word | center) = 0.03695
loss = 3.29814


## 4. Derive the gradients from the loss

A **derivative** tells us how the loss changes when we make one number slightly larger. A **gradient** collects those derivatives for all the model's weights. Gradient descent moves each weight a small distance in the opposite direction of its derivative.

For one training pair $(c,o)$, the model predicts every possible outside word $j$, but only $o$ was observed. Write $y_j=1$ if $j=o$, and $y_j=0$ otherwise. The vector $y\in\mathbb R^V$ is called a **one-hot target**.

### Step 1: rewrite the loss using the scores

Start with the original softmax (E3) and loss (E4). The logarithm turns the fraction into a difference, and $\log(e^x)=x$:

$$
\begin{aligned}
L(c,o)
&=-\log P(o\mid c)\\
&=-\log\left(\frac{e^{s(c,o)}}{\sum_{t=0}^{V-1}e^{s(c,t)}}\right)\\
&=-s(c,o)+\log\left(\sum_{t=0}^{V-1}e^{s(c,t)}\right).
\end{aligned}
$$

The first part rewards the observed word's score. The second part depends on **every** candidate score because all candidates share the softmax denominator.

### Step 2: differentiate with respect to one score

Choose a candidate $j$. The derivative of $-s(c,o)$ is $-1$ when $j=o$ and $0$ otherwise, which we write as $-y_j$. For the other part, use $\frac{d}{dx}\log x=1/x$ and $\frac{d}{dx}e^x=e^x$:

$$
\begin{aligned}
\frac{\partial[-s(c,o)]}{\partial s(c,j)}&=-y_j,\\
\frac{\partial}{\partial s(c,j)}
\log\left(\sum_{t=0}^{V-1}e^{s(c,t)}\right)
&=\frac{e^{s(c,j)}}{\sum_{t=0}^{V-1}e^{s(c,t)}}
=P(j\mid c).
\end{aligned}
$$

Adding these two derivatives gives the score gradient used in the code:

$$
g_j\equiv\frac{\partial L}{\partial s(c,j)}
=P(j\mid c)-y_j,
\qquad g\in\mathbb R^V.
\tag{E5}
$$

For intuition, suppose the three candidate probabilities are $[0.665,0.245,0.090]$ and the **second** word was observed. Then $y=[0,1,0]$ and $g=[0.665,-0.755,0.090]$. The observed word has a negative derivative, so a gradient-descent step tends to raise its score. The other two derivatives are positive, so the step tends to lower their scores.

### Step 3: follow the score back to $W_{\mathrm{out}}$

From (E2), $s(c,j)=\sum_{k=1}^{d}W_{\mathrm{out}}[j,k](h_c)_k$. Changing $W_{\mathrm{out}}[j,k]$ changes score $j$ by $(h_c)_k$. The **chain rule** multiplies this local effect by the loss's sensitivity to that score:

$$
\underbrace{\frac{\partial L}{\partial W_{\mathrm{out}}[j,k]}}_{\text{effect on loss}}
=\underbrace{\frac{\partial L}{\partial s(c,j)}}_{g_j}
\underbrace{\frac{\partial s(c,j)}{\partial W_{\mathrm{out}}[j,k]}}_{(h_c)_k}
=g_j(h_c)_k.
$$

Doing this for every row $j$ and coordinate $k$ gives an **outer product**:

$$
\nabla_{W_{\mathrm{out}}}L=gh_c^\mathsf{T}
\in\mathbb R^{V\times d}.
\tag{E6}
$$

In the code, `grad_out[j][k] = g[j] * w_in[center][k]` implements (E6).

### Step 4: follow all scores back to the center vector

The center coordinate $(h_c)_k$ appears in **every** score, not just one. For score $j$, its local effect is $\partial s(c,j)/\partial(h_c)_k=W_{\mathrm{out}}[j,k]$. Add the chain-rule contributions from all candidates:

$$
\begin{aligned}
\frac{\partial L}{\partial(h_c)_k}
&=\sum_{j=0}^{V-1}
\frac{\partial L}{\partial s(c,j)}
\frac{\partial s(c,j)}{\partial(h_c)_k}\\
&=\sum_{j=0}^{V-1}g_jW_{\mathrm{out}}[j,k],\\
\nabla_{h_c}L&=W_{\mathrm{out}}^\mathsf{T}g
\in\mathbb R^d.
\end{aligned}
\tag{E7}
$$

In the code, `grad_center[k] += g[j] * w_out[j][k]` implements the sum in (E7). Because $h_c$ is row $c$ of $W_{\mathrm{in}}$, only that input row receives this gradient; every other input row has gradient zero for this pair.

### Step 5: take a gradient-descent step

For learning rate $\eta>0$, subtract a small multiple of each gradient:

$$
\begin{aligned}
W_{\mathrm{in}}[c,k]&\leftarrow W_{\mathrm{in}}[c,k]
-\eta\frac{\partial L}{\partial(h_c)_k},\\
W_{\mathrm{out}}[j,k]&\leftarrow W_{\mathrm{out}}[j,k]
-\eta g_j(h_c)_k.
\end{aligned}
\tag{E8}
$$

The `step` function implements these two lines. It computes **both gradients from the old weights** before changing either matrix, so the derivatives match the same forward calculation.

In [5]:
def loss_and_gradients(center, outside, w_in, w_out):
    _, p = forward(center, w_in, w_out)  # E2–E3: scores and probabilities.
    loss = -math.log(p[outside])  # E4: observed-pair loss.
    V_local, d = len(w_out), len(w_in[0])
    g = [p[j] - (1.0 if j == outside else 0.0) for j in range(V_local)]  # E5.
    grad_out = [[g[j] * w_in[center][k] for k in range(d)]
                for j in range(V_local)]  # E6: V×d.
    grad_center = [0.0] * d
    for j in range(V_local):
        for k in range(d):
            grad_center[k] += g[j] * w_out[j][k]  # E7: use OLD W_out.
    return loss, grad_center, grad_out

loss, grad_center, grad_out = loss_and_gradients(c, o, example_in, example_out)
print('loss =', round(loss, 5))
print('gradient shapes: center =', len(grad_center),
      ', output =', len(grad_out), '×', len(grad_out[0]))

loss = 3.29814
gradient shapes: center = 8 , output = 27 × 8


## 5. Train by repeated gradient descent

An **epoch** is one pass through all training pairs. We shuffle the pairs each epoch and make one update per pair. This is **stochastic gradient descent**. The mean loss on the fixed pair list lets us see whether training is working. It may not fall after every individual update, but it should improve overall.

In [6]:
def step(center, outside, w_in, w_out, learning_rate):
    loss, grad_center, grad_out = loss_and_gradients(center, outside, w_in, w_out)  # E4–E7.
    d = len(w_in[0])
    for k in range(d):
        w_in[center][k] -= learning_rate * grad_center[k]  # E8: update center row.
    for j in range(len(w_out)):
        for k in range(d):
            w_out[j][k] -= learning_rate * grad_out[j][k]  # E8: update all output rows.
    return loss

def mean_loss(pairs, w_in, w_out):
    total = 0.0
    for center, outside in pairs:
        _, p = forward(center, w_in, w_out)  # E2–E3.
        total -= math.log(p[outside])  # E4.
    return total / len(pairs)  # E4: average over training pairs.

def train(pairs, V, d, epochs=60, learning_rate=0.08, seed=4):
    w_in, w_out = initialize(V, d, seed)  # E2: two parameter matrices.
    rng = random.Random(seed+100)
    history = [mean_loss(pairs, w_in, w_out)]
    for epoch in range(epochs):
        shuffled = pairs[:]
        rng.shuffle(shuffled)
        for center, outside in shuffled:
            step(center, outside, w_in, w_out, learning_rate)  # E4–E8.
        history.append(mean_loss(pairs, w_in, w_out))  # E4: monitor progress.
    return w_in, w_out, history

W_in, W_out, history = train(pairs, V, d=8)
print('epoch   mean training loss')
for epoch in (0, 10, 20, 40, 60):
    print(f'{epoch:5d}   {history[epoch]:.4f}')
assert history[-1] < history[0]

epoch   mean training loss
    0   3.2964
   10   1.9346
   20   1.0922
   40   1.0298
   60   1.0287


## 6. Inspect learned words

After training, row $i$ of $W_{\mathrm{in}}$ is the learned input embedding for word $i$. To compare two vectors $a,b\in\mathbb R^d$, use **cosine similarity**:

$$
\operatorname{cos}(a,b)
=\frac{a^\mathsf{T}b}
       {\|a\|_2\,\|b\|_2},
\qquad
\|a\|_2=\sqrt{\sum_{k=1}^{d}a_k^2}.
\tag{E9}
$$

Cosine compares vector directions rather than lengths. We use it **only to inspect** embeddings; the training objective was (E4), not cosine similarity. We also inspect the most likely context words under (E3), which directly reflect the model's prediction task. On a tiny corpus, neighbors can change with the random seed.

In [7]:
def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))  # E9: dot product.
    norm_a = math.sqrt(sum(x*x for x in a))  # E9: Euclidean length.
    norm_b = math.sqrt(sum(y*y for y in b))
    return dot/(norm_a*norm_b) if norm_a and norm_b else 0.0

def nearest(query, w_in, top_n=4):
    q = word_to_id[query]
    scored = [(word, cosine(w_in[q], w_in[i])) for word, i in word_to_id.items() if i != q]  # E9.
    return sorted(scored, key=lambda item: item[1], reverse=True)[:top_n]

for word in ('cat', 'apple', 'car'):
    _, p = forward(word_to_id[word], W_in, W_out)  # E2–E3: learned context distribution.
    best_ids = sorted(range(V), key=lambda j: p[j], reverse=True)[:4]
    print(word, 'nearest vectors:', [(w, round(s, 3)) for w, s in nearest(word, W_in)])
    print('    likely contexts:', [(id_to_word[j], round(p[j], 3)) for j in best_ids])

cat nearest vectors: [('dog', 1.0), ('puppy', 0.999), ('kitten', 0.999), ('milk', 0.755)]
    likely contexts: [('likes', 0.367), ('chases', 0.319), ('drinks', 0.301), ('sweet', 0.001)]
apple nearest vectors: [('banana', 0.999), ('orange', 0.999), ('pear', 0.999), ('good', 0.761)]
    likely contexts: [('tastes', 0.62), ('is', 0.37), ('road', 0.001), ('drives', 0.001)]
car nearest vectors: [('truck', 1.0), ('van', 1.0), ('bus', 0.999), ('road', 0.741)]
    likely contexts: [('has', 0.571), ('drives', 0.417), ('is', 0.002), ('wheels', 0.001)]


## 7. Experiment: change embedding dimension

Train the same model on the same pairs with $d=2$ and $d=8$. Keep the learning rate, 60 epochs, and random seed fixed. Since there are two $V\times d$ matrices, the number of learned scalars is

$$
N_{\mathrm{parameters}}=Vd+Vd=2Vd.
\tag{E10}
$$

The table compares initial and final **mean training loss**. Lower training loss does not prove better predictions on unseen text; a model with more parameters can overfit a small corpus.

In [8]:
print('dimension  parameters  initial loss  final loss  improvement')
experiment = []
for d in (2, 8):
    exp_in, exp_out, exp_history = train(pairs, V, d=d, epochs=60,
                                         learning_rate=0.08, seed=4)  # E2–E8.
    parameters = 2*V*d  # E10: two V×d matrices.
    improvement = exp_history[0] - exp_history[-1]  # E4: mean-loss change.
    experiment.append((d, exp_history[-1]))
    print(f'{d:9d}  {parameters:10d}  {exp_history[0]:12.4f}  '
          f'{exp_history[-1]:10.4f}  {improvement:11.4f}')
assert all(final < history[0] for _, final in experiment)

dimension  parameters  initial loss  final loss  improvement
        2         108        3.3020      1.3035       1.9985


        8         432        3.2964      1.0287       2.2677


## 8. Independent library check (NumPy only here)

The training code used Python loops. Now use vectorized NumPy operations **only to check** the result for one fixed pair. Let $s_c\in\mathbb R^V$ be the list of compatibility scores for center word $c$:

$$
\begin{aligned}
s_c &= W_{\mathrm{out}}h_c,\\
p_j &= P(j\mid c)=\operatorname{softmax}(s_c)_j,\\
L &= -\log p_o,\\
g &= p-y,\\
\nabla_{W_{\mathrm{out}}}L &= gh_c^\mathsf{T},\\
\nabla_{h_c}L &= W_{\mathrm{out}}^\mathsf{T}g.
\end{aligned}
$$

The shapes are $h_c\in\mathbb R^d$, $s_c,p,y,g\in\mathbb R^V$, $\nabla_{W_{\mathrm{out}}}L\in\mathbb R^{V\times d}$, and $\nabla_{h_c}L\in\mathbb R^d$. `np.allclose` compares the independently calculated loss and gradients with our loop-based implementation.

In [9]:
import numpy as np  # Only used as an independent check, never for training.

check_center, check_outside = pairs[7]
my_loss, my_grad_center, my_grad_out = loss_and_gradients(
    check_center, check_outside, W_in, W_out)  # E4–E7: our implementation.
np_in, np_out = np.array(W_in), np.array(W_out)
h = np_in[check_center]                    # E2: shape d.
scores = np_out @ h                       # E2: V compatibility scores.
exp_scores = np.exp(scores)               # E3: direct softmax numerator.
p = exp_scores / np.sum(exp_scores)       # E3: shape V.
reference_loss = -np.log(p[check_outside])  # E4.
y = np.zeros(V)
y[check_outside] = 1.0
g = p-y                                   # E5: shape V.
reference_grad_out = np.outer(g, h)       # E6: shape V×d.
reference_grad_center = np_out.T @ g      # E7: shape d.
assert np.allclose(my_loss, reference_loss, atol=1e-10)
assert np.allclose(my_grad_center, reference_grad_center, atol=1e-10)
assert np.allclose(my_grad_out, reference_grad_out, atol=1e-10)
print('PASS: loss, center gradient, and every output gradient match NumPy.')

PASS: loss, center gradient, and every output gradient match NumPy.


## 9. What to say in the presentation

**Problem (1 min):** how can words become numbers that reflect their contexts? **Idea (1 min):** show `cat likes milk` and its center/context pairs. **Math (4 min):** walk from the $V\times d$ matrices through dot products (E2), softmax (E3), loss (E4), chain-rule gradients (E5–E7), and update (E8). **Demo (3 min):** show the training loss, dimension experiment table, and final check. **Limitations (1 min):** tiny data, one fixed vector per word (no separate meanings of `bank`), possible stereotypes in text, and $O(Vd)$ work per pair with full softmax. Both teammates should practice every equation and code cell for questions.

**Takeaway:** this is a multi-step AI training pipeline that you implemented yourselves: examples → scores → probabilities → loss → gradients → repeated optimization → embeddings.